<h1>Workflow for Transcriptomic RNA-seq Data</h1>

This notebook details workflow for Illumina paired-end transcriptomics data. This includes quality assessment, read trimming and filtering, assembly, mapping, building expression matrix, and functional annotation. Other analysis steps (DESeq2 and WGCNA) are discussed in RScripts located in the parent folder.

---
## <font color ='blue'>How to Use This Notebook</font>

1. This notebook assumes that Trinity was installed using Docker. Before opening this notebook, make sure that Daemon is enabled.
>`sudo service docker start`
2. Then, open jupyter notebook with the command below and select the notebook.
>`jupyter notebook`
3. To run the cells in this notebook, press Shift+Enter.

---
## Tools Used
1. <b>Trinity (Docker version)</b>. Installation instructions can be found [here](https://github.com/trinityrnaseq/trinityrnaseq/wiki/Trinity-in-Docker).
2. <b>FastQC</b>. Can be installed through bioconda.
3. <b>Trimmomatic</b>. Can be installed through bioconda.

---
## Starting Files 

1. This Jupyter notebook

---
## Acknowledgement
The data used for this demonstration are from 6 samples taken from the following study: [Mary et al. (2022)](https://www.microbiologyresearch.org/content/journal/mgen/10.1099/mgen.0.000879):

<i>Mary, L., Quere, J., Latimier, M., Rovillon, G. A., Hégaret, H., Réveillon, D., & Le Gac, M. (2022). Genetic association of toxin production in the dinoflagellate Alexandrium minutum. Microbial Genomics, 8(11), 000879.</i>

---
## Table of Contents
 * [**Step 1: Data Preparation**](#Step-1:-Data-Preparation)  
     * [Prepare directories](#Prepare-directories)
     * [Download data](#Download-data)
     * [Rename and move the data files (optional)](#Rename-and-move-the-data-files-(optional))  
 * [**Step 2: Quality Control**](#Step-2:-Quality-Control)  
     * [Inspect data quality](#Inspect-data-quality)
     * [Trim and filter](#Trim-and-filter)
     * [Inspect data quality post Trimmomatic](#Inspect-data-quality-post-Trimmomatic)
 * [**Step 3: Assembly (optional)**](#Step-3:-Assembly-(optional))
     * [Generate transcriptome assembly](#Generate-transcriptome-assembly)
     * [Taxonomy assignment](#Taxonomy-assignment)
     * [Exporting OTU tables](#Exporting-OTU-tables)
 * [**Step 4: Mapping**](#Step-4:-Mapping)
     * [Index the reference assembly](#Index-the-reference-assembly)
     * [Map reads to assembly](#Map-reads-to-assembly)
 * [**Step 5: Building Expression Matrix**](#Step-5:-Building-Expression-Matrix)
     * [Prepare quant files](#Prepare-quant-file)
     * [Generate expression matrix](#Generate-expression-matrix)
 * [**Step 6: Functional Annotation**](#Step-6:-Functional-Annotation)
     * [TransDecoder + Trinotate](#TransDecoder-+-Trinotate)
     * [BLAST](#BLAST)
     * [Diamond](#Diamond)
     * [eggNOG mapper](#eggNOG-mapper)
---

# <font color = 'gray'>Step 1: Data Preparation</font>

### Prepare directories

Generate the folders where output files of the subsequent steps will be stored.

In [ ]:
%%bash

mkdir \
    0-raw_data \
    1-pre_trim_fastqc \
    2-trimmomatic \
    3-post_trim_fastqc \
    4-assembly_trinity \
    5-mapping \
    6-build_matrix

### Download data

The data that will be used for the demonstration of this workflow was taken from the study mentioned in the Acknowledgement section. Run the following cell to download the raw data from ENA database. Replace the links below if you want to process a different dataset.

In [ ]:
cmd = '''
wget \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/003/ERR9606893/ERR9606893_1.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/003/ERR9606893/ERR9606893_2.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/005/ERR9606895/ERR9606895_1.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/005/ERR9606895/ERR9606895_2.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/006/ERR9606896/ERR9606896_1.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/006/ERR9606896/ERR9606896_2.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/003/ERR9606933/ERR9606933_1.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/003/ERR9606933/ERR9606933_2.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/004/ERR9606934/ERR9606934_1.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/004/ERR9606934/ERR9606934_2.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/006/ERR9606936/ERR9606936_1.fastq.gz' \
    'ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR960/006/ERR9606936/ERR9606936_2.fastq.gz' \
    -P ./0-raw_data
'''

!{cmd}

### Rename and move the data files (optional)

To keep things neat and easily trackable, let's rename the FASTQ files.

In [ ]:
%%bash

mv ./0-raw_data/ERR9606893_1.fastq.gz ./0-raw_data/T1xT2_beta_1.fastq.gz
mv ./0-raw_data/ERR9606893_2.fastq.gz ./0-raw_data/T1xT2_beta_2.fastq.gz
mv ./0-raw_data/ERR9606895_1.fastq.gz ./0-raw_data/T1xT2_delta_1.fastq.gz
mv ./0-raw_data/ERR9606895_2.fastq.gz ./0-raw_data/T1xT2_delta_2.fastq.gz
mv ./0-raw_data/ERR9606896_1.fastq.gz ./0-raw_data/T1xT2_epsilon_1.fastq.gz
mv ./0-raw_data/ERR9606896_2.fastq.gz ./0-raw_data/T1xT2_epsilon_2.fastq.gz
mv ./0-raw_data/ERR9606933_1.fastq.gz ./0-raw_data/NT1xNT2_VII_1.fastq.gz
mv ./0-raw_data/ERR9606933_2.fastq.gz ./0-raw_data/NT1xNT2_VII_2.fastq.gz
mv ./0-raw_data/ERR9606934_1.fastq.gz ./0-raw_data/NT1xNT2_VIII_1.fastq.gz
mv ./0-raw_data/ERR9606934_2.fastq.gz ./0-raw_data/NT1xNT2_VIII_2.fastq.gz
mv ./0-raw_data/ERR9606936_1.fastq.gz ./0-raw_data/NT1xNT2_X_1.fastq.gz
mv ./0-raw_data/ERR9606936_2.fastq.gz ./0-raw_data/NT1xNT2_X_2.fastq.gz

In [ ]:
%%bash

mkdir 0-raw_data/T1xT2_{beta,delta,epsilon} 0-raw_data/NT1xNT2_{VII,VIII,X}

for spl in $(ls -d ./0-raw_data/*/ | xargs -n 1 basename)
do
    mv ./0-raw_data/${spl}_1.fastq.gz ./0-raw_data/${spl}
    mv ./0-raw_data/${spl}_2.fastq.gz ./0-raw_data/${spl}
done

# <font color = 'gray'>Step 2: Quality Control</font>

### Inspect data quality

Before anything else, you must first assess the quality of the raw data. This will help you evaluate whether the data's quality is ideal for the type of analysis you will be conducting. 

After the run completes, check the HTML reports inside `1-pre_trim_fastqc`. A few things to check in the reports:

<ul>
    <li>Number of reads. (i.e. <i>What is the sample depth?</i>)</li>
    <li>Per-base sequence quality. (i.e. <i>How good/bad is the quality of the sequences?</i>)</li>
    <li>Adapter content. (i.e. <i>Do the reads contain adapter sequences? If yes, what adapter sequences?</i>)</li>
</ul>


>**NOTE:** In the line:`conda run -n qc-env --live-stream`, replace `qc-env` by the name of the conda environment where FastQC is installed. If it is installed in the base environment, you can remove this line.


In [ ]:
cmd = '''
for spl in $(ls -d ./0-raw_data/*/ | xargs -n 1 basename)
do
    echo -e "\n===== Running FastQC on: ${spl} =====\n"
    mkdir ./1-pre_trim_fastqc/${spl}
    
    conda run -n qc-env --live-stream \
        fastqc \
            --outdir ./1-pre_trim_fastqc/${spl} \
            ./0-raw_data/${spl}/*
done
'''

!{cmd}

### Trim and filter

After initial inspection, you can now trim and filter the raw sequencing reads. The list below describes the Trimmomatic input and output arguments.

-  `./0-raw_data/${spl}/*1.fastq.gz` - Raw forward reads.
-  `./0-raw_data/${spl}/*2.fastq.gz` - Raw reverse reads.
-  `./2-trimmomatic/${spl}/${spl}_ofp.fastq.gz` - OFP = **O**utput **F**orward **P**aired; Trimmed and filtered forward reads that still have a pair
-  `./2-trimmomatic/${spl}/${spl}_ofu.fastq.gz` - OFU = **O**utput **F**orward **U**npaired; Trimmed and filtered forward reads that no longer has a pair
-  `./2-trimmomatic/${spl}/${spl}_orp.fastq.gz` - ORP = **O**utput **R**everse **P**aired; Trimmed and filtered reverse reads that still have a pair
-  `./2-trimmomatic/${spl}/${spl}_oru.fastq.gz` - ORU = **O**utput **R**everse **U**npaired; Trimmed and filtered reverse reads that no longer has a pair


Additionally, according the the quality report, some sequences still contain adapters. You can remove those by using the `ILLUMINACLIP` option and specifiying the FASTA file containing the adapter sequences (e.g. In this case, `TruSeq3-PE.fa`). Check what library preparation method you utilized. You can find some FASTA files of adapter sequences [here](https://github.com/timflutre/trimmomatic/tree/master/adapters). 

Other Trimmomamatic parameters are described in this [manual](http://www.usadellab.org/cms/uploads/supplementary/Trimmomatic/TrimmomaticManual_V0.32.pdf). Adjust the parameters to whatever suits your data and objective. For instance, if you want to analyze SNPs, you may need to apply more stringent parameters to ensure that your data is of high quality, albeit, this may lead to more reads being dropped.

>**NOTE:** In the line:`conda run -n qc-env --live-stream`, replace `qc-env` by the name of the conda environment where Trimmomatic is installed. If it is installed in the base environment, you can remove this line.

>**NOTE:** The cell below iterates through all samples. If you will be using your own data, make sure that the folders in `0-raw_folder` correspond to your sample names.

In [ ]:
cmd = '''
for spl in $(ls -d ./0-raw_data/*/ | xargs -n 1 basename)
do
    echo -e "\n===== Trimming sample: ${spl} =====\n"
    mkdir 2-trimmomatic/${spl}
    
    conda run -n qc-env --live-stream \
        trimmomatic PE \
            ./0-raw_data/${spl}/*1.fastq.gz \
            ./0-raw_data/${spl}/*2.fastq.gz \
            ./2-trimmomatic/${spl}/${spl}_ofp.fastq.gz \
            ./2-trimmomatic/${spl}/${spl}_ofu.fastq.gz \
            ./2-trimmomatic/${spl}/${spl}_orp.fastq.gz \
            ./2-trimmomatic/${spl}/${spl}_oru.fastq.gz \
            ILLUMINACLIP:TruSeq3-PE.fa:2:30:10:8:true \
            LEADING:3 \
            TRAILING:3 \
            MAXINFO:80:0.8 \
            MINLEN:70 2>&1 | tee -a ./2-trimmomatic/${spl}/${spl}_trimmomatic.log
done
'''

!{cmd}

### Inspect data quality post Trimmomatic

For sanity check, we'll inspect once again if the sequencing data has been trimmed and filtered appropriately. Check the HTML reports and compare those to the raw sequencing data.


>**NOTE:** In the line:`conda run -n qc-env --live-stream`, replace `qc-env` by the name of the conda environment where FastQC is installed. If it is installed in the base environment, you can remove this line.


In [ ]:
cmd = '''
for spl in $(ls -d ./0-raw_data/*/ | xargs -n 1 basename)
do
    echo -e "\n===== Running FastQC on: ${spl} =====\n"
    mkdir ./3-post_trim_fastqc/${spl}
    
    conda run -n genome_assembly2 --live-stream \
        fastqc \
            --outdir ./3-post_trim_fastqc/${spl} \
            ./2-trimmomatic/${spl}/*o?p.fastq.gz
done
'''

!{cmd}

# <font color = 'gray'>Step 3: Assembly (optional)</font>

### Generate transcriptome assembly

This step performs <i>de novo</i> assembly using the clean reads. If a high-quality genome assembly data is available, you may opt to do a reference-guided assembly as discussed [here](https://github.com/trinityrnaseq/trinityrnaseq/wiki/Genome-Guided-Trinity-Transcriptome-Assembly). Alternatively, if a reference transcriptome already exists, you can opt to proceed to the next step.

You can run the code below, however, this can be quite resource-intensive. As an alternative, you can run Trinity for free using the [Galaxy webserver](https://usegalaxy.eu/).

The FASTA headers of the output assembly of Trinity is explained [here](https://github.com/trinityrnaseq/trinityrnaseq/wiki/Output-of-Trinity-Assembly).

In [ ]:
cmd = '''
docker run --rm -v `pwd`:`pwd` trinityrnaseq/trinityrnaseq \
Trinity \
    --left $(ls -1 `pwd`/0-raw_data/*/*1.fastq.gz | paste -sd "," -) \
    --right $(ls -1 `pwd`/0-raw_data/*/*2.fastq.gz | paste -sd "," -) \
    --seqType fq \
    --max_memory 6G \
    --CPU 4 \
    --output `pwd`/4-assembly_trinity
'''

!{cmd}

# <font color = 'gray'>Step 4: Mapping</font>

### Index the reference assembly

Before, mapping we should prepare our assembly file by indexing it.

>**NOTE:** If you are using a pre-existing reference assembly, make sure to replace the filename of the assembly file (i.e. `a_minutum_ref_assembly.fasta`) indicated in the `--transcripts` parameter.

In [ ]:
cmd = '''
docker run --rm -v `pwd`:`pwd` trinityrnaseq/trinityrnaseq \
/usr/local/bin/util/align_and_estimate_abundance.pl \
    --transcripts `pwd`/4-assembly_trinity/a_minutum_ref_assembly.fasta \
    --est_method RSEM \
    --aln_method bowtie2 \
    --prep_reference
'''

!{cmd}

### Map reads to assembly

Now, using RSEM, we can map the clean reads per sample to the assembly. Again, make sure to replace the value passed to the `--transcripts` argument to whatever is applicable in you case.

>**NOTE:** If you want to analyze your transcripts closer to gene level than isoform level, you can use the `--gene_trans_map` or `--trinity_mode`. The former requires a tab-separated file of gene-to-transcript mapping. The latter automatically generates a gene-to-transcript mapping from your Trinity assembly.

In [ ]:
cmd = '''
for spl in $(ls -d ./0-raw_data/*/ | xargs -n 1 basename)
do
    mkdir ./5-mapping/${spl}
    echo -e "===== Mapping ${spl} to the reference assembly =====\n"

    docker run --rm -v `pwd`:`pwd` trinityrnaseq/trinityrnaseq \
    /usr/local/bin/util/align_and_estimate_abundance.pl \
        --transcripts `pwd`/4-assembly_trinity/a_minutum_ref_assembly.fasta \
        --seqType fq \
        --left `pwd`/2-trimmomatic/${spl}/*ofp.fastq.gz \
        --right `pwd`/2-trimmomatic/${spl}/*orp.fastq.gz \
        --est_method RSEM \
        --aln_method bowtie2 \
        --output_dir `pwd`/5-mapping/${spl}/rsem 2>&1 | tee -a `pwd`/5-mapping/${spl}/${spl}_est_abund.log
done
'''

!{cmd}

# <font color = 'gray'>Step 5: Building Expression Matrix</font>

###  Prepare quant file

The command below generates a file containing the directories of the mapping results.

>**NOTE:** Replace `RSEM.isoforms.results` by `RSEM.genes.results` if you want to analyze at gene level. Gene-level summary is only possible if you specified a gene-to-transcript mapping in the previous step.

In [ ]:
%%bash

for spl in $(ls -d ./0-raw_data/*/ | xargs -n 1 basename)
do
    mv ./5-mapping/${spl}/rsem/RSEM.isoforms.results ./5-mapping/${spl}/rsem/${spl}.isoforms.results
    mv ./5-mapping/${spl}/rsem/RSEM.genes.results ./5-mapping/${spl}/rsem/${spl}.genes.results
done

In [ ]:
%%bash

for spl in $(ls -d ./0-raw_data/*/ | xargs -n 1 basename)
do
    realpath $(find ./5-mapping/${spl} -maxdepth 2 -name "${spl}.isoforms.results") | tee -a 6-build_matrix/rsem_quant_outputs.txt
done

### Generate expression matrix

Finally, you can produce your feature count table (i.e. transcripts vs samples). This can be used as input to subsequent analysis like DESeq2 (differential expression analysis), and WGCNA (weighted gene co-expression network analysis).

In [ ]:
cmd = '''
docker run --rm -v `pwd`:`pwd` trinityrnaseq/trinityrnaseq \
/usr/local/bin/util/abundance_estimates_to_matrix.pl \
    --name_sample_by_basedir \
    --basedir_index -3 \
    --est_method RSEM \
    --gene_trans_map none \
    --quant_files `pwd`/6-build_matrix/rsem_quant_outputs.txt \
    --out_prefix `pwd`/6-build_matrix/rsem_exp_matrix
'''

!{cmd}

# <font color = 'gray'>Step 6: Functional Annotation</font>

Annotation of the transcripts put biological meaning to the assembled sequences that you currently have. 

This can be a very resource-intensive task especially if the transcriptome of interest is large. Depending on your goal, you can, however, find ways to decrease the computing resources needed. For example, if your'e only interested in differentially expressed transcripts/genes (DEG), you may opt to annotate only the transcripts that were found to be differentially expressed. That way, you're able to reduce the number of sequences needing annotation. However, for some objectives like over-representation analysis, you'll likely need to annotate all of your reference transcripts.

The approach you will apply in this step will be dependent on several factors like type of organism you are studying, annotation rates (i.e. how many transcripts have hits), etc. You can play around with different methods and databases to find what fits best with your goals. For instance, if you are searching for functions that are already well-described, you can use the highly-curated Swiss-Prot database, with some tool like BLAST to find sequence similarities. However, a typical drawback when using a highly-curated database is that you may have lower annotation rates. If you find that this does not suit your needs, you can try other databases or make a custom one.

Listed below are some annotation approaches that you can explore.

### TransDecoder + Trinotate

Trinity has a pipeline for annotation (TransDecoder + Trinotate). TransDecoder predicts the coding regions in the assembled transcripts, while Trinotate can be used to functionally annotate the predicted coding regions. Note that these two tools do not need to be executed as a pair. You can use TransDecoder to predict coding regions and utilize a different homology search tool to functionally annotate those coding regions.

TransDecoder could be executed in two steps which is outlined [here](https://github.com/griffithlab/rnaseq_tutorial/wiki/Trinotate-Functional-Annotation#identification-of-likely-protein-coding-regions-in-transcripts).

Afterwards, you can run the subsequent steps up until the generation of Trinotate report as discussed [here](https://github.com/griffithlab/rnaseq_tutorial/wiki/Trinotate-Functional-Annotation#sequence-homology-searches)

An easier alternative is to run it in [Galaxy EU webserver](https://usegalaxy.eu/).

One downside of this approach is that if your assembly is not very contiguous, TransDecoder may fail to identify coding regions in many reads. Thus, many reads will be dropped and excluded for annotation. If that happens in your case, you may opt not to run TransDecoder.

### BLAST

BLAST is one of the most common bioinformatics tool widely used to find sequence homology. If you will be using this method, another important aspect to consider is the database of sequences to which you are comparing your query sequences against.

Common databases:
-  **NCBI nt database**
    -  Non-redundant nucleotide database. 
    -  Pros:
        1. Vastness of database will likely yield more hits. 
    -  Cons:
        1. Difficult to run on a local device due to computer resource limitations (i.e. Large database size, likely longer runtimes, etc).
        2. Level of curation of database.
-  **Swiss-Prot**
    -  Protein database manually curated by experts. 
    -  Pros:
        1. Reliability due to level of curation
        2. Cross-references (Gene Ontology, MetaCyc, KEGG, Pfam, etc) makes it easier to analyze at different levels of function (i.e. gene level vs pathway level). 
    -  Cons:
        1. Lower number of reference sequences will likely result to lower annotation rates
-  **RefSeq**
    -  NCBI’s Reference Sequence (RefSeq) database is a collection of taxonomically diverse, non-redundant and richly annotated sequences representing naturally occurring molecules of DNA, RNA, and protein. Included are sequences from plasmids, organelles, viruses, archaea, bacteria, and eukaryotes. (source: https://www.ncbi.nlm.nih.gov/books/NBK21091/). You can find taxonomically categorized RefSeq entries [here](https://ftp.ncbi.nlm.nih.gov/genomes/refseq/).
    -  Pros: 
        1. Has separate categories for different taxonomic lineages which allows a more targeted annotation.
    -  Cons:
        1. Can be biased on model organisms (which is also probably true, to some extent, in most databases).
-  **Custom database**
    -  You can also create your own database. For instance, you can pull out annotated genomes and/or transcriptomes of organisms closely related to your target species from [JGI](https://genome.jgi.doe.gov/portal/) and use this as your reference database.
    -  Pros:
        1. More targeted.
    -  Cons:
        1. More pre-processing steps depending on the source reference genome or transcriptome.


BLAST is available in Galaxy webserver alongside some of these databases, albeit, most of these databases are not the latest updates.

### Diamond

Diamond offers a very similar functionality to BLAST. If you need a significantly faster runtime with a bit of a loss in sensitivity, Diamond is the way to go. You can use the same databases mentioned above.

### eggNOG mapper

EggNOG-mapper is a tool for fast functional annotation of novel sequences. It uses precomputed orthologous groups and phylogenies from the eggNOG database to transfer functional information from fine-grained orthologs only.

Common uses of eggNOG-mapper include the annotation of novel genomes, transcriptomes or even metagenomic gene catalogs.

The use of orthology predictions for functional annotation permits a higher precision than traditional homology searches (i.e. BLAST searches), as it avoids transferring annotations from close paralogs (duplicate genes with a higher chance of being involved in functional divergence).

(Source: https://github.com/eggnogdb/eggnog-mapper)

Besides accuracy of functional inference, one advantage of eggNOG are cross-references. The output table of eggNOG-mapper does not only display hits of query sequences with its database's orthologous groups, but also information on associated functional categories (GO, KEGG, Pfam, etc.). A portion of an example output table is shown below. However, a downside, based on previous experiences, is that it can sometimes yield low annotation rates, likely because the database mostly consists of well-described groups only.

eggNOG-mapper is hosted in [EMBL](http://eggnog-mapper.embl.de/), as well as in Galaxy webserver.

**eggNOG-mapper Example Output**

| query | seed_ortholog | evalue | score | eggNOG_OGs | max_annot_lvl | COG_category | Description | Preferred_name | GOs | ... |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| comp100001_c0_seq1.p1 | 3067.XP_002946533.1 | 3.37e-17 | 89.4 | KOG2998@1,root,KOG2998@2759,Eukaryota,37KCB@33090,Viridiplantae,34MG7@3041,Chlorophyta | 3041,Chlorophyta | S | ELMO/CED-12 family | - | - | ... |
| comp100003_c0_seq1.p2 | 1547445.LO80_05060 | 2.6e-86 | 269 | COG3491@1,root,COG3491@2,Bacteria,1MUNT@122,Proteobacteria,1RPQ8@1236,Gammaproteobacteria,460D6@72273,Thiotrichales | 72273,Thiotrichales | C | Belongs to the iron ascorbate-dependent oxidoreductase family | - | - | ... |
| comp100008_c0_seq1.p1 | 6087.XP_002162980.2 | 2.05e-16 | 92.8 | KOG2242@1,root,KOG2242@2759,Eukaryota,38FNX@33154,Opisthokonta,3BD0H@33208,Metazoa | 33208,Metazoa | A | heterogeneous nuclear ribonucleoprotein | HNRNPU | GO:0000122,GO:0000166,... | ... |